# Train on Colab GPU from a mounted Parquet dataset

**Architecture:** all preprocessing (feature engineering + standardization + lagged responders + clustering) is done **once, locally**, and exported to a partitioned Parquet dataset that lives on Google Drive (`scripts/export_parquet.py`). This notebook does **no** Kaggle download and **no** feature build — it just mounts Drive and trains on the GPU. Every session starts training in ~2 minutes.

**One-time local setup** (on your laptop):
```bash
uv run python scripts/precompute_dataset.py --min-date 700 --max-date 1698 \
    --train-until 1598 --lagged-responders 0,1,2,3,4,5,6,7,8 --out artifacts/precomputed/pool700_lags
uv run python scripts/export_parquet.py \
    --memmap artifacts/precomputed/pool700_lags --out artifacts/precomputed/pool700_lags_parquet
```
then copy `pool700_lags_parquet/` **and** the repo folder to Google Drive.

**Runtime → Change runtime type → GPU** before running.

In [ ]:
!nvidia-smi -L
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Mount Drive
Everything persistent (data + code + results) lives on Drive, so nothing is re-uploaded between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Point at the mounted repo + parquet dataset
Edit these two paths to wherever you put them on Drive.

In [ ]:
from pathlib import Path
REPO = Path('/content/drive/MyDrive/janestreet/project')          # the repo folder
DATA = Path('/content/drive/MyDrive/janestreet/pool700_lags_parquet')  # the parquet dataset
OUT  = Path('/content/drive/MyDrive/janestreet/results')          # results persist here
OUT.mkdir(parents=True, exist_ok=True)
assert (DATA/'manifest.json').exists(), 'parquet dataset not found at DATA'
assert (REPO/'pyproject.toml').exists(), 'repo not found at REPO'

## 3. Install the package (editable) + deps
`pip install -e` off Drive works; if Drive I/O is slow you can `!cp -r $REPO /content/project` first and install from there.

In [ ]:
%pip install --quiet 'einops>=0.8' 'numpy>=2.0' 'polars>=1.20' 'pyarrow>=15' \
  'scikit-learn>=1.5' 'torch>=2.4' 'tqdm>=4.66' 'xgboost>=2.1' 'pyyaml>=6' 'iisignature>=0.24'
%pip install --quiet -e $REPO
%cd $REPO

In [ ]:
# Sanity: the mounted dataset is self-describing — check the schema without any rebuild.
import json
m = json.loads((DATA/'manifest.json').read_text())
print('format   :', m.get('format'))
print('features :', m['K'], 'cols   lags:', m.get('lagged_responders'), ' signals:', m.get('responder_signal_features'))
print('dates    :', min(int(d) for d in m['date_row_ranges']), '..', max(int(d) for d in m['date_row_ranges']),
      ' train_until:', m['train_until'])
print('partitions:', len(list((DATA/'data').glob('date_id=*'))))

## 4. Train on GPU
Same scripts as local — the only change is `--data` points at the mounted parquet and `--device cuda --batch-size 16` uses the GPU. Results write straight back to Drive.

**Fair recency sweep** (`gru_modelr`, converged 30-epoch budget) — the whole thing is ~2 h on a T4:

In [ ]:
!KMP_DUPLICATE_LIB_OK=TRUE python scripts/sweep_window_length.py \
  --data $DATA --model gru_modelr \
  --lengths 50,100,200,400,898 --hi 1597 --pool-lo 700 \
  --valid-lo 1599 --valid-hi 1698 \
  --device cuda --batch-size 16 --epochs 30 --patience 6 \
  --out $OUT/window_sweep

**Or a single model / bag member** (uncomment) — trains one window and dumps its online predictions for blending back into the laptop ensemble:

In [ ]:
# !KMP_DUPLICATE_LIB_OK=TRUE python scripts/train_from_memmap.py \
#   --data $DATA --model gru_modelr --resample-mode window \
#   --train-lo 1318 --train-hi 1597 --valid-lo 1599 --valid-hi 1698 \
#   --device cuda --batch-size 16 --epochs 30 --patience 6 \
#   --tag gpu_280d --out $OUT/single

## 5. Results
Everything is already on Drive under `results/` — no download step. Peek at the recency curve:

In [ ]:
import json
s = json.load(open(OUT/'window_sweep'/'sweep_summary.json'))
print('window   static      online')
for r in s['rows']:
    print(f"{r['length']:>5}d  {r['r2_static']:+.5f}  {r['r2_online']:+.5f}")